In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded! ✅")

Libraries loaded! ✅


In [2]:
df = pd.read_csv('/Users/abhinavjha/NeuralRetail/data/processed/clean_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Data loaded: {df.shape} ✅")

Data loaded: (397924, 9) ✅


In [3]:
snapshot_date = df['InvoiceDate'].max()
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
    }).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm['Churn'] = (rfm['Recency'] > 90).astype(int)
print(f"RFM shape: {rfm.shape} ✅")
rfm.head()

RFM shape: (4339, 5) ✅


,CustomerID,Recency,Frequency,Monetary,Churn
0,12346.0,325,1,77183.60,1
1,12347.0,1,7,4310.00,0
2,12348.0,74,4,1797.24,0
3,12349.0,18,1,1757.55,0
4,12350.0,309,1,334.40,1


In [4]:
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred)
print(f"AUC-ROC Score: {auc:.4f} ✅")
print(classification_report(y_test, model.predict(X_test)))

AUC-ROC Score: 1.0000 ✅
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       569
           1       1.00      1.00      1.00       299

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



In [5]:
import joblib
joblib.dump(model, '/Users/abhinavjha/NeuralRetail/models/churn_model.pkl')
print("Churn model saved! ✅")

Churn model saved! ✅
